# Saher — توليد فيديو من النص باستخدام Wan2.1

دفتر Google Colab لتوليد مقطع قصير من وصف إنجليزي باستخدام التطبيق الرسمي المفتوح المصدر [Wan2.1](https://github.com/Wan-Video/Wan2.1) والأوزان العامة [Wan-AI/Wan2.1-T2V-1.3B](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B). لا يحتاج إلى مفتاح API أو خادم عام.

**قبل البدء:** شغّل الخلايا بالترتيب من أعلى إلى أسفل. قد يستغرق تنزيل الأوزان عدة دقائق، ويعتمد التوليد على GPU المتاح في جلستك. نموذج 1.3B موجه هنا إلى 480p (832×480)، وهي الدقة التي يوصي بها المشروع لهذا النموذج.

## 1. اختيار GPU في Google Colab

من قائمة Colab اختر **Runtime → Change runtime type → T4 GPU** (أو أي GPU متاح)، ثم اضغط **Save**. على الهاتف قد تحتاج إلى فتح قائمة Colab أو تفعيل عرض سطح المكتب في المتصفح لرؤية إعداد نوع التشغيل. بعد الاتصال، ابدأ تشغيل الخلايا بالترتيب.

الوصول المجاني إلى GPU غير مضمون؛ قد تنفد الحصة أو لا يتوفر GPU في الوقت الحالي. عندها انتظر ثم أعد المحاولة. جلسات Colab مؤقتة وقد تنقطع، لذلك نزّل الفيديو إلى الهاتف فور اكتماله.

In [ ]:
# 2. التحقق من GPU باستخدام nvidia-smi
import shutil
import subprocess
import torch

if not shutil.which("nvidia-smi"):
    raise RuntimeError("لم يتم العثور على nvidia-smi. اختر GPU من Runtime > Change runtime type ثم أعد الاتصال.")
result = subprocess.run(["nvidia-smi"], check=False)
if result.returncode != 0 or not torch.cuda.is_available():
    raise RuntimeError("لا يظهر GPU لـ PyTorch. أعد اختيار GPU وتأكد من اتصال الجلسة قبل المتابعة.")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("BF16 supported:", torch.cuda.is_bf16_supported())

## 2. استنساخ مشروع Wan2.1 الرسمي

تنفّذ الخلية التالية استنساخًا سطحيًا للفرع `main` من مستودع Wan-Video الرسمي داخل مساحة Colab المؤقتة.

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/Wan2.1")
if not (REPO_DIR / "generate.py").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Wan-Video/Wan2.1.git", str(REPO_DIR)],
        check=True,
    )
else:
    print("Wan2.1 موجود مسبقًا في", REPO_DIR)
print("المسار:", REPO_DIR)

## 3. تثبيت الاعتماديات

تعليمات المشروع الرسمية تتطلب Python 3.10 أو أحدث وPyTorch 2.4.0 أو أحدث، وتستخدم `pip install -r requirements.txt`. في Colab نستخدم قائمة الاعتماديات الرسمية مع استثناء `flash_attn` الاختياري، لأنه امتداد CUDA قد يتطلب بناءً غير متوافق مع صورة Colab؛ يختار كود Wan2.1 تلقائيًا مسار PyTorch SDPA المدمج عند غيابه. لا نثبت Gradio أو نفتح أي خادم للمستخدم، رغم وجود الحزمة ضمن قائمة المشروع.

لا تُعد تثبيت PyTorch يدويًا إذا كان إصدار Colab الحالي يفي بالمتطلبات.

In [ ]:
import sys
import subprocess
from pathlib import Path
from packaging.version import Version

if sys.version_info < (3, 10):
    raise RuntimeError(f"يتطلب Wan2.1 Python >= 3.10؛ الموجود: {sys.version}")

requirements = (REPO_DIR / "requirements.txt").read_text().splitlines()
colab_requirements = [
    line for line in requirements
    if not line.strip().lower().startswith("flash_attn")
]
filtered_path = Path("/content/wan2.1-requirements-colab.txt")
filtered_path.write_text("\n".join(colab_requirements) + "\n")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(filtered_path)],
    check=True,
)
import torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
if Version(torch.__version__.split("+")[0]) < Version("2.4.0"):
    raise RuntimeError("يتطلب المشروع PyTorch 2.4.0 أو أحدث. استخدم بيئة Colab محدثة.")
if not torch.cuda.is_available():
    raise RuntimeError("لم يعد GPU متاحًا. أعد الاتصال بـ GPU ثم أعد تشغيل هذا الدفتر من البداية.")

## 4. تنزيل النموذج الرسمي

الأوزان متاحة للعامة في Hugging Face، ولا يتطلب تنزيلها تسجيل دخول أو مفتاحًا في الاستخدام المعتاد. سيتم حفظ الملفات في مجلد checkpoint الذي يتوقعه أمر Wan2.1. حجم التنزيل كبير، وقد يعتمد الزمن على اتصال Colab.

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B"
MODEL_DIR = REPO_DIR / "Wan2.1-T2V-1.3B"
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(MODEL_DIR),
)
print("اكتمل التنزيل:", MODEL_DIR)

## 5. إنشاء الفيديو

عدّل `PROMPT` باللغة الإنجليزية كما تريد، ثم شغّل الخلية. تستعمل الأمر الرسمي `generate.py --task t2v-1.3B --size 832*480` مع `--sample_shift 8` و`--sample_guide_scale 6`. خيارات `--offload_model True` و`--t5_cpu` تقلل استهلاك VRAM، لكنها قد تجعل التوليد أبطأ. يكتب الملف الناتج إلى `/content/saher.mp4`.

In [ ]:
import subprocess
import sys
from pathlib import Path

PROMPT = (
    "Cinematic fantasy scene, a young wizard in an enchanted forest raises a glowing "
    "crystal staff, blue magical energy spirals around it, sparks illuminate ancient "
    "trees, dramatic camera push-in, smooth motion, detailed atmosphere, professional "
    "fantasy cinematography."
)
OUTPUT = Path("/content/saher.mp4")
command = [
    sys.executable, "generate.py",
    "--task", "t2v-1.3B",
    "--size", "832*480",
    "--ckpt_dir", str(MODEL_DIR),
    "--offload_model", "True",
    "--t5_cpu",
    "--sample_shift", "8",
    "--sample_guide_scale", "6",
    "--save_file", str(OUTPUT),
    "--prompt", PROMPT,
]
print("بدء التوليد؛ قد يستغرق وقتًا طويلًا بحسب GPU المتاح...")
subprocess.run(command, cwd=str(REPO_DIR), check=True)
if not OUTPUT.is_file() or OUTPUT.stat().st_size == 0:
    raise RuntimeError("لم يُنشأ ملف الفيديو المتوقع.")
print(f"تم إنشاء {OUTPUT} ({OUTPUT.stat().st_size / 1024**2:.1f} MiB)")

## 6. معاينة الفيديو وتنزيله

إذا اكتمل التوليد بنجاح، تعرض الخلية التالية معاينة داخل Colab. وقدّم الخلية الأخيرة زر تنزيل الملف إلى الهاتف.

In [ ]:
from IPython.display import Video, display

display(Video("/content/saher.mp4", embed=True, width=480))

In [ ]:
from google.colab import files

files.download("/content/saher.mp4")

## استكشاف الأخطاء وإصلاحها

- **GPU غير موجود:** افتح **Runtime → Change runtime type** واختر GPU، ثم أعد الاتصال. لا تضمن Colab المجانية توفر GPU أو استمرار الحصة.
- **أخطاء CUDA أو تعذر تهيئة GPU:** تأكد من أن الخلية الأولى تعرض GPU وأن `torch.cuda.is_available()` صحيح. أعد تشغيل الجلسة، ثم نفّذ الخلايا من البداية. تجنب تثبيت نسخة Torch عشوائية فوق نسخة Colab.
- **نفاد VRAM:** تستخدم الخلية إعداد 832×480 وخيارات نقل النموذج إلى CPU. أعد تشغيل الجلسة وأغلق عمليات GPU الأخرى إن وجدت. لا يمكن ضمان نجاح كل أنواع GPU المجانية، وقد لا تتناسب بعض البطاقات أو الجلسات مع متطلبات النموذج.
- **فشل تنزيل الأوزان:** تحقق من اتصال الإنترنت وأعد تشغيل خلية التنزيل. المستودع عام ولا يحتاج عادةً إلى رمز دخول. إذا طلب Hugging Face تسجيل دخول بسبب تغيير في سياسة الوصول أو تقييد مؤقت، سجّل الدخول داخل Colab فقط عند الحاجة ولا تضع أي سر في الدفتر أو المحادثة.
- **تعارض الحزم أو تثبيت بطيء:** انتظر حتى تنتهي عملية `pip`. إذا طلب Colab إعادة تشغيل بيئة التشغيل، أعد تشغيلها ونفّذ الخلايا من البداية. يستثني هذا الدفتر `flash_attn` الاختياري، ويستخدم مسار الانتباه المدمج في PyTorch.
- **انقطاع الجلسة:** ملفات `/content` مؤقتة وقد تُحذف عند انتهاء الجلسة. أعد تشغيل الخلايا من البداية لتنزيل النموذج وإنشاء الفيديو مرة أخرى، ونزّل `saher.mp4` إلى الهاتف فور ظهوره.

هذا الدفتر لم يُختبر بتوليد فعلي على GPU داخل هذه البيئة؛ يلزم تشغيله في Colab متصل ببطاقة مناسبة. نجاح الفحص الثابت لا يضمن توافر GPU أو نجاح التوليد في كل جلسة.

## المصادر

- [تعليمات Wan2.1 الرسمية](https://github.com/Wan-Video/Wan2.1)
- [بطاقة وأوزان Wan-AI/Wan2.1-T2V-1.3B](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B)
- [واجهة تنزيل Hugging Face Hub](https://huggingface.co/docs/huggingface_hub/guides/download)